# Whisper Audio Transcription Demo

All three protocols: REST JSON, gRPC, REST Binary

**Important:** Run cells in order. REST JSON must run first to avoid tritonclient state issues.

In [ ]:
import os
import time
import numpy as np
import librosa
import tritonclient.grpc as grpcclient
import tritonclient.http as httpclient
from transformers import WhisperProcessor

# =============================================================================
# Configuration - Update NAMESPACE for your environment
# =============================================================================
NAMESPACE = "domino-inference-dev"
#NAMESPACE = "domino-inference-test"
#NAMESPACE = "domino-inference-prod"

os.environ["TRITON_GRPC_URL"] = f"triton-inference-server-proxy.{NAMESPACE}.svc.cluster.local:50051"
os.environ["TRITON_REST_URL"] = f"http://triton-inference-server-proxy.{NAMESPACE}.svc.cluster.local:8080"

GRPC_URL = os.environ.get("TRITON_GRPC_URL", "localhost:50051")
REST_URL = os.environ.get("TRITON_REST_URL", "http://localhost:8080")

API_KEY = os.environ.get("DOMINO_USER_API_KEY", "")
grpc_headers = {"x-domino-api-key": API_KEY} if API_KEY else None
http_headers = {"X-Domino-Api-Key": API_KEY} if API_KEY else None

print(f"Namespace: {NAMESPACE}")
print(f"gRPC URL: {GRPC_URL}")
print(f"REST URL: {REST_URL}")
print(f"Auth: {'API Key configured' if API_KEY else 'Disabled'}")

In [ ]:
# =============================================================================
# Prepare inputs
# =============================================================================
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")

audio_path = "../samples/audio_sample.wav"
audio, sr = librosa.load(audio_path, sr=16000, mono=True)
print(f"Audio duration: {len(audio)/sr:.2f} seconds")

url = REST_URL.replace("http://", "").replace("https://", "")

In [ ]:
# =============================================================================
# 1. REST (JSON) - MUST RUN FIRST
# =============================================================================
inputs_json = processor(audio, sampling_rate=16000, return_tensors="np")
features_json = inputs_json.input_features.astype(np.float32).copy()

client = httpclient.InferenceServerClient(url=url)
inp = httpclient.InferInput("input_features", list(features_json.shape), "FP32")
inp.set_data_from_numpy(features_json, binary_data=False)
out = httpclient.InferRequestedOutput("transcription", binary_data=False)

start = time.time()
resp = client.infer("whisper-tiny-python", [inp], outputs=[out], headers=http_headers)
t_json = (time.time() - start) * 1000
trans_json = resp.as_numpy("transcription")[0]
if isinstance(trans_json, bytes):
    trans_json = trans_json.decode("utf-8")

print(f"✓ REST (JSON):   {t_json:8.2f} ms")

In [ ]:
# =============================================================================
# 2. gRPC Protocol
# =============================================================================
inputs_grpc = processor(audio, sampling_rate=16000, return_tensors="np")
features_grpc = inputs_grpc.input_features.astype(np.float32).copy()

client = grpcclient.InferenceServerClient(url=GRPC_URL)
inp = grpcclient.InferInput("input_features", list(features_grpc.shape), "FP32")
inp.set_data_from_numpy(features_grpc)
out = grpcclient.InferRequestedOutput("transcription")

start = time.time()
resp = client.infer("whisper-tiny-python", [inp], outputs=[out], headers=grpc_headers)
t_grpc = (time.time() - start) * 1000
trans_grpc = resp.as_numpy("transcription")[0]
if isinstance(trans_grpc, bytes):
    trans_grpc = trans_grpc.decode("utf-8")
client.close()

print(f"✓ gRPC:          {t_grpc:8.2f} ms")

In [ ]:
# =============================================================================
# 3. REST (Binary) Protocol
# =============================================================================
inputs_binary = processor(audio, sampling_rate=16000, return_tensors="np")
features_binary = inputs_binary.input_features.astype(np.float32).copy()

client = httpclient.InferenceServerClient(url=url)
inp = httpclient.InferInput("input_features", list(features_binary.shape), "FP32")
inp.set_data_from_numpy(features_binary, binary_data=True)
out = httpclient.InferRequestedOutput("transcription", binary_data=False)

start = time.time()
resp = client.infer("whisper-tiny-python", [inp], outputs=[out], headers=http_headers)
t_binary = (time.time() - start) * 1000
trans_binary = resp.as_numpy("transcription")[0]
if isinstance(trans_binary, bytes):
    trans_binary = trans_binary.decode("utf-8")

print(f"✓ REST (Binary): {t_binary:8.2f} ms")

In [ ]:
# =============================================================================
# Results
# =============================================================================
print(f"Transcription (JSON):   {trans_json}")
print(f"Transcription (gRPC):   {trans_grpc}")
print(f"Transcription (Binary): {trans_binary}")